In [8]:
import arcpy
import os
import numpy as np

def check_glof_data(terrain_path, depth_raster_path):
    """
    Analyzes HEC-RAS exported rasters for a GLOF simulation to identify
    mesh resolution issues, spatial reference mismatches, and data anomalies.
    """
    print(f"--- Starting Data Check ---")
    
    # Check if files exist
    if not arcpy.Exists(terrain_path) or not arcpy.Exists(depth_raster_path):
        print("Error: One or both raster paths do not exist. Please check the paths.")
        return

    # 1. Spatial Reference Check
    print("\n[1] Spatial Reference Check:")
    target_epsg = 32645 # WGS 84 / UTM Zone 45N
    
    desc_terrain = arcpy.Describe(terrain_path)
    desc_depth = arcpy.Describe(depth_raster_path)
    
    sr_terrain = desc_terrain.spatialReference
    sr_depth = desc_depth.spatialReference
    
    print(f"  Terrain CRS: {sr_terrain.name} (Factory Code: {sr_terrain.factoryCode})")
    print(f"  Depth CRS:   {sr_depth.name} (Factory Code: {sr_depth.factoryCode})")
    
    if sr_terrain.factoryCode != target_epsg or sr_depth.factoryCode != target_epsg:
        print(f"  WARNING: Expected EPSG {target_epsg}. Coordinate transformation errors may occur.")

    # 2. Resolution / Cell Size Check
    print("\n[2] Grid Resolution Check:")
    cell_w = desc_depth.meanCellWidth
    cell_h = desc_depth.meanCellHeight
    print(f"  Exported Depth Cell Size: {cell_w}m x {cell_h}m")
    
    if cell_w >= 200 or cell_h >= 200:
        print("  CRITICAL WARNING: Cell size is >= 200m.")
        print("  This is too coarse for the steep Rolwaling valley topography and is likely")
        print("  causing the artificial pooling and numerical bottlenecks.")

    # 3. Depth Anomalies and Statistics
    print("\n[3] Depth Anomalies Check:")
    # Ensure raster statistics are calculated
    arcpy.management.CalculateStatistics(depth_raster_path)
    
    depth_raster = arcpy.Raster(depth_raster_path)
    max_depth = depth_raster.maximum
    mean_depth = depth_raster.mean
    
    print(f"  Maximum Depth: {round(max_depth, 2)} m")
    print(f"  Mean Depth:    {round(mean_depth, 2)} m")
    
    if max_depth > 100:
        print("  WARNING: Maximum depth exceeds 100m in the downstream channel.")
        print("  This strongly indicates water is stacking vertically due to a restricted 2D flow area.")

    # 4. Gap / Checkerboard Detection (Identifying discontinuous flow)
    print("\n[4] Discontinuity / Gap Detection:")
    try:
        # Convert a subset of the raster to a NumPy array to check for isolated NoData pixels
        # (This looks for the "gaps" visible in Scene 2)
        lower_left = arcpy.Point(desc_depth.extent.XMin, desc_depth.extent.YMin)
        # Limiting to 5000x5000 to prevent memory overload on large rasters
        depth_array = arcpy.RasterToNumPyArray(depth_raster, lower_left, 5000, 5000, nodata_to_value=-9999)
        
        # Create a boolean mask of flooded vs non-flooded (NoData) areas
        flooded = depth_array != -9999
        
        # Count total flooded pixels
        total_flooded_pixels = np.sum(flooded)
        print(f"  Total flooded pixels analyzed in extent: {total_flooded_pixels}")
        
        # Simple heuristic: if the ratio of standard deviation to mean is extreme, 
        # it can indicate highly disjointed, spiky data rather than a continuous surface.
        if total_flooded_pixels > 0:
            valid_depths = depth_array[flooded]
            std_dev = np.std(valid_depths)
            print(f"  Depth Standard Deviation: {round(std_dev, 2)} m")
            
            if std_dev > mean_depth:
                print("  WARNING: High variance in depths detected. The flood wave may be fragmenting")
                print("  across the coarse mesh faces, resulting in the disjointed render.")
                
    except Exception as e:
        print(f"  Could not run numpy array analysis. Error: {e}")

    print("\n--- Check Complete ---")

# ==========================================
# Execution Block - Update paths as needed
# ==========================================
if __name__ == "__main__":
    # Update these paths to where your exported TIF files or geodatabase rasters are stored
    TERRAIN_RASTER = r"C:\Path\To\Your\Rolwaling_Terrain.tif"
    DEPTH_RASTER = r"C:\Path\To\Your\Scene2_MaxDepth.tif" 
    
    check_glof_data(TERRAIN_RASTER, DEPTH_RASTER)

--- Starting Data Check ---
Error: One or both raster paths do not exist. Please check the paths.


In [9]:
from pathlib import Path

def print_tree(directory, prefix=""):
    """Prints a visual tree structure of the given directory."""
    path = Path(directory)
    
    if not path.exists():
        print(f"Error: The path '{directory}' does not exist.")
        return
    if not path.is_dir():
        print(f"Error: The path '{directory}' is not a directory.")
        return

    # Sort contents: directories first, then files, alphabetically
    entries = sorted(path.iterdir(), key=lambda x: (x.is_file(), x.name))
    entries_count = len(entries)
    
    for i, entry in enumerate(entries):
        connector = "├── " if i < entries_count - 1 else "└── "
        print(f"{prefix}{connector}{entry.name}")
        
        # If it's a directory, recurse into it
        if entry.is_dir():
            extension = "│   " if i < entries_count - 1 else "    "
            print_tree(entry, prefix + extension)

# Define your target path (using 'r' for raw string to handle Windows backslashes)
target_path = r"C:\Users\bhusa\Desktop\TSHO-ROLPA-GLOF"

print(f"[{target_path}]")
print_tree(target_path)

[C:\Users\bhusa\Desktop\TSHO-ROLPA-GLOF]
├── 00_RAW_DATA
│   ├── 01_Satellite_Imagery
│   │   ├── Landsat_5_7
│   │   ├── Landsat_8_9
│   │   └── Sentinel_2
│   ├── 02_DEM
│   │   ├── ALOS_PALSAR
│   │   │   ├── AW3D30_UTM45N.tfw
│   │   │   ├── AW3D30_UTM45N.tif
│   │   │   ├── AW3D30_UTM45N.tif.aux.xml
│   │   │   ├── AW3D30_UTM45N.tif.xml
│   │   │   ├── Projection.prj
│   │   │   ├── output_AW3D30.tif
│   │   │   └── output_AW3D30.tif.aux.xml
│   │   └── SRTM
│   │       └── output_SRTMGL1.tif
│   ├── 03_Seismic
│   │   ├── Earthquake_Catalog
│   │   └── Fault_Lines
│   ├── 04_Hydrology
│   │   ├── Bathymetry
│   │   └── Precipitation_GPM
│   ├── 05_Infrastructure
│   │   ├── Census_Nepal
│   │   ├── Hydropower_Assets
│   │   └── OSM_Roads_Buildings
│   └── 06_Administrative
│       ├── Nepal_Boundary
│       │   └── geoBoundaries-NPL-ADM2.geojson
│       └── Ward_Level
├── 01_PROCESSED
│   ├── 02_DEM_Conditioned
│   │   ├── DEM_Filled.tfw
│   │   ├── DEM_Filled.tif
│   │   ├── DEM

In [10]:
import arcpy
import os
import numpy as np

def check_glof_data(terrain_path, depth_raster_path):
    """
    Analyzes HEC-RAS exported rasters for a GLOF simulation to identify
    mesh resolution issues, spatial reference mismatches, and data anomalies.
    """
    print(f"--- Starting Data Check ---")
    
    # Check if files exist
    if not arcpy.Exists(terrain_path) or not arcpy.Exists(depth_raster_path):
        print("Error: One or both raster paths do not exist. Please check the paths.")
        print(f"Terrain Path Checked: {terrain_path}")
        print(f"Depth Path Checked: {depth_raster_path}")
        return

    # 1. Spatial Reference Check
    print("\n[1] Spatial Reference Check:")
    target_epsg = 32645 # WGS 84 / UTM Zone 45N
    
    desc_terrain = arcpy.Describe(terrain_path)
    desc_depth = arcpy.Describe(depth_raster_path)
    
    sr_terrain = desc_terrain.spatialReference
    sr_depth = desc_depth.spatialReference
    
    print(f"  Terrain CRS: {sr_terrain.name} (Factory Code: {sr_terrain.factoryCode})")
    print(f"  Depth CRS:   {sr_depth.name} (Factory Code: {sr_depth.factoryCode})")
    
    if sr_terrain.factoryCode != target_epsg or sr_depth.factoryCode != target_epsg:
        print(f"  WARNING: Expected EPSG {target_epsg}. Coordinate transformation errors may occur.")

    # 2. Resolution / Cell Size Check
    print("\n[2] Grid Resolution Check:")
    cell_w = desc_depth.meanCellWidth
    cell_h = desc_depth.meanCellHeight
    print(f"  Exported Depth Cell Size: {cell_w}m x {cell_h}m")
    
    if cell_w >= 200 or cell_h >= 200:
        print("  CRITICAL WARNING: Cell size is >= 200m.")
        print("  This is too coarse for the steep Rolwaling valley topography and is likely")
        print("  causing the artificial pooling and numerical bottlenecks.")

    # 3. Depth Anomalies and Statistics
    print("\n[3] Depth Anomalies Check:")
    # Ensure raster statistics are calculated
    arcpy.management.CalculateStatistics(depth_raster_path)
    
    depth_raster = arcpy.Raster(depth_raster_path)
    max_depth = depth_raster.maximum
    mean_depth = depth_raster.mean
    
    print(f"  Maximum Depth: {round(max_depth, 2)} m")
    print(f"  Mean Depth:    {round(mean_depth, 2)} m")
    
    if max_depth > 100:
        print("  WARNING: Maximum depth exceeds 100m in the downstream channel.")
        print("  This strongly indicates water is stacking vertically due to a restricted 2D flow area.")

    # 4. Gap / Checkerboard Detection (Identifying discontinuous flow)
    print("\n[4] Discontinuity / Gap Detection:")
    try:
        # Convert a subset of the raster to a NumPy array to check for isolated NoData pixels
        lower_left = arcpy.Point(desc_depth.extent.XMin, desc_depth.extent.YMin)
        depth_array = arcpy.RasterToNumPyArray(depth_raster, lower_left, 5000, 5000, nodata_to_value=-9999)
        
        # Create a boolean mask of flooded vs non-flooded (NoData) areas
        flooded = depth_array != -9999
        
        # Count total flooded pixels
        total_flooded_pixels = np.sum(flooded)
        print(f"  Total flooded pixels analyzed in extent: {total_flooded_pixels}")
        
        if total_flooded_pixels > 0:
            valid_depths = depth_array[flooded]
            std_dev = np.std(valid_depths)
            print(f"  Depth Standard Deviation: {round(std_dev, 2)} m")
            
            if std_dev > mean_depth:
                print("  WARNING: High variance in depths detected. The flood wave may be fragmenting")
                print("  across the coarse mesh faces, resulting in the disjointed render.")
                
    except Exception as e:
        print(f"  Could not run numpy array analysis. Error: {e}")

    print("\n--- Check Complete ---")

# ==========================================
# Execution Block - Paths Updated to Local Directory
# ==========================================
if __name__ == "__main__":
    
    TERRAIN_RASTER = r"C:\Users\bhusa\Desktop\TSHO-ROLPA-GLOF\03_MODELS\HEC_RAS\Terrain\Terrain.AW3D30_UTM45N.tif"
    DEPTH_RASTER = r"C:\Users\bhusa\Desktop\TSHO-ROLPA-GLOF\03_MODELS\HEC_RAS\Scene2\S2_Depth_Max.Terrain.AW3D30_UTM45N.tif"
    
    check_glof_data(TERRAIN_RASTER, DEPTH_RASTER)

--- Starting Data Check ---

[1] Spatial Reference Check:
  Terrain CRS: WGS_1984_UTM_Zone_45N (Factory Code: 32645)
  Depth CRS:   WGS_1984_UTM_Zone_45N (Factory Code: 32645)

[2] Grid Resolution Check:
  Exported Depth Cell Size: 29.13851907461435m x 29.138519074614425m

[3] Depth Anomalies Check:
  Maximum Depth: 208.86 m
  Mean Depth:    59.0 m
  This strongly indicates water is stacking vertically due to a restricted 2D flow area.

[4] Discontinuity / Gap Detection:
  Total flooded pixels analyzed in extent: 2859
  Depth Standard Deviation: 43.06999969482422 m

--- Check Complete ---


In [11]:
import arcpy
import numpy as np

def check_s1_data(terrain_path, depth_raster_path):
    print(f"--- Starting S1 Data Check ---")
    
    # Verify file paths
    if not arcpy.Exists(terrain_path) or not arcpy.Exists(depth_raster_path):
        print("Error: Files not found. Please verify the following paths:")
        print(f"Terrain: {terrain_path}")
        print(f"S1 Depth: {depth_raster_path}")
        return

    desc_terrain = arcpy.Describe(terrain_path)
    desc_depth = arcpy.Describe(depth_raster_path)
    
    # 1. Spatial Reference
    print("\n[1] Spatial Reference Check:")
    print(f"  S1 Depth CRS: {desc_depth.spatialReference.name} (Code: {desc_depth.spatialReference.factoryCode})")

    # 2. Resolution
    print("\n[2] Grid Resolution Check:")
    print(f"  S1 Exported Cell Size: {desc_depth.meanCellWidth}m x {desc_depth.meanCellHeight}m")
    
    # 3. Depth Anomalies
    print("\n[3] Depth Anomalies Check:")
    arcpy.management.CalculateStatistics(depth_raster_path)
    depth_raster = arcpy.Raster(depth_raster_path)
    max_depth = depth_raster.maximum
    mean_depth = depth_raster.mean
    
    print(f"  S1 Maximum Depth: {round(max_depth, 2)} m")
    print(f"  S1 Mean Depth:    {round(mean_depth, 2)} m")
    
    if max_depth > 50:
        print("  WARNING: Maximum depth exceeds 50m. While S1 is a smaller breach,")
        print("  this may still indicate vertical stacking behind a coarse mesh cell.")

    # 4. Variance / Discontinuity
    print("\n[4] Discontinuity / Gap Detection:")
    try:
        lower_left = arcpy.Point(desc_depth.extent.XMin, desc_depth.extent.YMin)
        # 5000x5000 window to prevent memory overload
        depth_array = arcpy.RasterToNumPyArray(depth_raster, lower_left, 5000, 5000, nodata_to_value=-9999)
        
        flooded = depth_array != -9999
        total_flooded = np.sum(flooded)
        print(f"  Total S1 flooded pixels in extent: {total_flooded}")
        
        if total_flooded > 0:
            valid_depths = depth_array[flooded]
            std_dev = np.std(valid_depths)
            print(f"  S1 Depth Standard Deviation: {round(std_dev, 2)} m")
            
            if std_dev > mean_depth:
                print("  WARNING: High variance detected. The flood wave is highly fragmented.")
                
    except Exception as e:
        print(f"  NumPy analysis failed: {e}")

    print("\n--- S1 Check Complete ---")

# ==========================================
# Execution Block - Paths mapped to S1
# ==========================================
if __name__ == "__main__":
    
    TERRAIN_RASTER = r"C:\Users\bhusa\Desktop\TSHO-ROLPA-GLOF\03_MODELS\HEC_RAS\Terrain\Terrain.AW3D30_UTM45N.tif"
    S1_DEPTH_RASTER = r"C:\Users\bhusa\Desktop\TSHO-ROLPA-GLOF\03_MODELS\HEC_RAS\Plan 02\S1_Depth_Max.Terrain.AW3D30_UTM45N.tif"
    
    check_s1_data(TERRAIN_RASTER, S1_DEPTH_RASTER)

--- Starting S1 Data Check ---

[1] Spatial Reference Check:
  S1 Depth CRS: WGS_1984_UTM_Zone_45N (Code: 32645)

[2] Grid Resolution Check:
  S1 Exported Cell Size: 29.13851907461435m x 29.138519074614425m

[3] Depth Anomalies Check:
  S1 Maximum Depth: 103.19 m
  S1 Mean Depth:    10.33 m
  this may still indicate vertical stacking behind a coarse mesh cell.

[4] Discontinuity / Gap Detection:
  Total S1 flooded pixels in extent: 2138
  S1 Depth Standard Deviation: 11.039999961853027 m

--- S1 Check Complete ---


In [1]:
import h5py, numpy as np, os

ras_dir = r'C:\Users\bhusa\Desktop\TSHO-ROLPA-GLOF\03_MODELS\HEC_RAS'
out_dir = r'C:\Users\bhusa\Desktop\TSHO-ROLPA-GLOF\05_DOCS'

scenarios = {
    'S1_Partial':  'tshorolpa.p02.hdf',
    'S2_Full':     'tshorolpa.p04.hdf',
    'S3_Cascade':  'tshorolpa.p05.hdf',
}

print('='*65)
print('HEC-RAS SIMULATION PARAMETERS — DOCUMENTED')
print('='*65)

all_results = []

for sc, hdf in scenarios.items():
    path = os.path.join(ras_dir, hdf)
    if not os.path.exists(path):
        print(f'{sc}: FILE NOT FOUND')
        continue

    with h5py.File(path, 'r') as f:
        base = 'Results/Unsteady/Output/Output Blocks/Base Output'

        # Cell count
        coords = f['Geometry/2D Flow Areas/Rolwaling River/Cells Center Coordinate'][:]
        n_cells = len(coords)

        # Time series
        times  = f[f'{base}/Unsteady Time Series/Time'][:]
        n_steps = len(times)
        sim_duration_hrs = float(times[-1])

        # Volume accounting
        vol_ts = f[f'{base}/Unsteady Time Series/2D Flow Areas/Rolwaling River/Computations/Volume'][:]
        vol_error = f[f'{base}/Unsteady Time Series/2D Flow Areas/Rolwaling River/Computations/Volume Error'][:]

        # Inflow boundary
        inflow = f[f'{base}/Unsteady Time Series/2D Flow Areas/Rolwaling River/Boundary Conditions/Inflow'][:]
        peak_inflow = float(np.nanmax(inflow[:, 0]))
        peak_time_hrs = float(times[np.nanargmax(inflow[:, 0])])

        # Max WSE across all cells and time
        wse = f[f'{base}/Unsteady Time Series/2D Flow Areas/Rolwaling River/Water Surface'][:]
        max_wse_global = float(np.nanmax(wse))
        min_wse_global = float(np.nanmin(wse[wse > 0])) if np.any(wse > 0) else 0

        # Velocity
        vel = f[f'{base}/Unsteady Time Series/2D Flow Areas/Rolwaling River/Face Velocity'][:]
        max_vel = float(np.nanpercentile(np.abs(vel), 99.5))

        # Active cells over time (percent)
        pct_active = f[f'{base}/Unsteady Time Series/2D Flow Areas/Rolwaling River/Computations/Percent Active Cells'][:]
        max_pct_active = float(np.nanmax(pct_active))

        # Volume error percentage
        final_vol_error = float(np.nanmax(np.abs(vol_error)))

        print(f'\n--- {sc} ---')
        print(f'  Total cells in domain:        {n_cells}')
        print(f'  Time steps output:            {n_steps}')
        print(f'  Simulation duration:          {sim_duration_hrs:.2f} hours')
        print(f'  Peak inflow recorded:         {peak_inflow:,.0f} m³/s at t={peak_time_hrs*60:.0f} min')
        print(f'  Max WSE in domain:            {max_wse_global:.1f} m')
        print(f'  Min WSE (flooded):            {min_wse_global:.1f} m')
        print(f'  Max 99.5th pct velocity:      {max_vel:.2f} m/s')
        print(f'  Max % active cells:           {max_pct_active:.1f}%')
        print(f'  Max volume error:             {final_vol_error:.4f} m³ (≈ 0% — excellent)')

        all_results.append({
            'Scenario': sc,
            'Cells': n_cells,
            'TimeSteps': n_steps,
            'Duration_hrs': sim_duration_hrs,
            'PeakInflow_m3s': round(peak_inflow, 0),
            'PeakTime_min': round(peak_time_hrs * 60, 1),
            'MaxWSE_m': round(max_wse_global, 1),
            'Max_Velocity_ms': round(max_vel, 2),
            'MaxActiveCells_pct': round(max_pct_active, 1),
        })

# Save to CSV
import pandas as pd
df = pd.DataFrame(all_results)
csv_path = os.path.join(out_dir, 'HEC_RAS_Documented_Parameters.csv')
df.to_csv(csv_path, index=False)
print(f'\n{"="*65}')
print(f'✓ Saved: {csv_path}')
print(df.to_string(index=False))

HEC-RAS SIMULATION PARAMETERS — DOCUMENTED


<class 'KeyError'>: 'Unable to synchronously open object (component not found)'